# LionAG2: Recursive Exploratory Research with AG2 beta — Typed multi-agent handoff (3/10)

In the [Day 2 tutorial](02_typed_findings.ipynb), we explored how to use AG2 to create structured output. To recap, the agent was granted an Exa search tool and was asked to create a structured output as a `Finding` with citations.

The structured output was handy for downstream data manipulation and adding conditions to agentic workflows. In today's tutorial, we will look into how to use structured output to create a multi-agent handoff workflow.

## Setup

In [1]:
import os
import asyncio

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.tools import ExaToolkit

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)
exa_tool = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

The schema from before was quite straightforward — produce a finding with citations from search results. Let's add some richer schemas:

In [2]:
class OpenQuestion(BaseModel):
    question: str = Field(description="A specific unresolved question surfaced by the survey.")
    novelty: float = Field(
        ge=0.0, le=1.0,
        description="0 = well-studied, 1 = barely explored.",
    )


class Survey(BaseModel):
    """Surveyor output: landscape of a topic plus open frontiers."""
    topic: str
    overview: str = Field(description="2-3 sentence summary of the current state of knowledge.")
    key_findings: list[str] = Field(description="Major established results (3-5 bullets).")
    open_questions: list[OpenQuestion] = Field(
        description="Unresolved questions ranked by novelty. At least 2.",
    )

`OpenQuestion` carries a `novelty` score — this will become the key signal for deciding which questions are worth drilling deeper into. A score of 0.7+ means active frontier, worth investigating. Anything below 0.4 is textbook material we can skip.

In [3]:
class Hypothesis(BaseModel):
    """Theorist output: one falsifiable claim with a test."""
    claim: str = Field(description="One-sentence falsifiable claim.")
    mechanism: str = Field(description="Why this claim might hold — the causal story.")
    testable_prediction: str = Field(
        description="An observable that would confirm or refute the claim."
    )
    confidence: float = Field(ge=0.0, le=1.0, description="Subjective confidence in the claim.")
    source_question: str = Field(description="The open question this hypothesis addresses.")

## Pipeline

The pipeline today will go as follows: a surveyor agent will conduct a survey via search tools over literature and produce a `Survey` object with open questions. For each open question, we will launch one theorist agent to research further and create a hypothesis.

In [4]:
topic = "high-Tc superconductivity"

surveyor = Agent(
    name="surveyor",
    prompt=(
        f"You are a meticulous research surveyor. Search broadly on {topic}"
        ", then summarize what is known and what remains open. Be specific "
        "about novelty — textbook material is 0.0, active frontiers are 0.7+."
    ),
    config=config,
    response_schema=Survey,
    tools=[exa_tool],
)

survey_reply = await surveyor.ask(
    "Survey the current state of high-Tc superconductivity research."
)
survey = await survey_reply.content()

The survey is a Pydantic object of type `Survey(BaseModel)` as defined above. Since we have more than one open question, we can launch parallel theorists to investigate them at the same time.

In [5]:
async def hypothesize(question: OpenQuestion, survey=survey) -> Hypothesis:
    theorist = Agent(
        name="theorist",
        prompt=(
            "You are a theoretical physicist. Given a survey and an open question, "
            "produce a precise, falsifiable hypothesis. Be concrete about the "
            "mechanism and what experiment would test it."
        ),
        config=config,
        response_schema=Hypothesis,
        tools=[exa_tool],
    )

    theorist_prompt = (
        f"Survey topic: {survey.topic}\n"
        f"Overview: {survey.overview}\n\n"
        f"Key findings:\n"
        + "\n".join(f"- {kf}" for kf in survey.key_findings)
        + f"\n\nFocus on this open question (novelty={question.novelty:.2f}):\n"
        f"{question.question}\n\n"
        f"Produce a falsifiable hypothesis with a concrete testable prediction."
    )

    hyp_reply = await theorist.ask(theorist_prompt)
    return await hyp_reply.content()

results = await asyncio.gather(*(hypothesize(q) for q in survey.open_questions))

Each theorist gets its own agent instance and the `asyncio.gather` runs them all concurrently.

In [6]:
for i, hyp in enumerate(results):
    lines = [
        f"### Hypothesis {i+1}",
        f"**Claim:** {hyp.claim}",
        f"**Mechanism:** {hyp.mechanism}",
        f"**Test:** {hyp.testable_prediction}",
        f"**Confidence:** {hyp.confidence:.2f}",
        f"**Addresses:** {hyp.source_question}",
    ]
    display(Markdown("\n\n".join(lines)))

### Hypothesis 1

**Claim:** In cuprates, the superconducting pairing glue is short-range antiferromagnetic exchange acting on doped Mott-oxide planes, and the pseudogap is a distinct competing spin/charge-ordered state that onsets from the same exchange scale but is not the pairing gap itself; therefore, superconductivity is strongest where charge order is weakest and where the spin-fluctuation spectrum retains strong Q=(c0,c0) weight.

**Mechanism:** Doping a CuO2 Mott insulator leaves a large nearest-neighbor exchange J that favors singlet formation and d-wave pairing through spin-fluctuation exchange. The pseudogap arises when the same exchange-driven correlations partially gap the antinodal states via incipient density-wave or local singlet correlations, but these states do not contribute coherently to zero-resistance pairing. As a result, the superconducting dome reflects a competition: pairing amplitude grows with residual AF exchange and carrier mobility, while static or quasi-static charge order depletes the antinodal density of states and suppresses phase coherence. Hole- and electron-doped families differ mainly in how strongly charge order preempts coherence, not in the basic pairing interaction.

**Test:** A unified experiment combining momentum-resolved electron spectroscopy and tunable perturbations should show that, at fixed doping, the superconducting gap scale tracks the integrated low-energy spin-fluctuation spectral weight near (c0,c0), while the pseudogap scale tracks the onset temperature/strength of charge-order correlations and can be independently suppressed or enhanced by uniaxial strain or field without producing a proportional change in the pairing gap. Specifically, if charge order is weakened, Tc should rise but the spin-fluctuation peak energy should remain approximately unchanged; if the pseudogap were the pairing gap, suppressing charge order would instead collapse both the pseudogap and superconducting gap together.

**Confidence:** 0.62

**Addresses:** What is the microscopic origin of pairing in cuprates, and how does it relate to the pseudogap, charge order, and strange-metal normal state across hole- and electron-doped families?

### Hypothesis 2

**Claim:** In hydrogen-rich clathrate hydrides, the pressure needed to stabilize a high-Tc phonon-driven superconducting phase can be reduced substantially if the hydrogen sublattice forms a precompressed, cage-like network around a multivalent metal whose valence electron count pins the Fermi level to a broad van Hove/flat phonon-coupling peak; specifically, candidate compounds with higher host-lattice precompression and stronger H-H connectivity will retain Tc > 100 K at significantly lower pressure than chemically similar hydrides with the same H content but less cage-like local structure.

**Mechanism:** The key mechanism is structural precompression: a cage or clathrate motif shortens H-H distances and stiffens the lattice so that the high-frequency hydrogen phonons and strong electron-phonon matrix elements survive at lower external pressure. The multivalent metal acts mainly as an electron donor and structural stabilizer, while the best-performing motifs are those that maximize hydrogen connectivity without driving the system into molecular H2 formation or lattice collapse. If this is correct, the decisive control parameter is not simply stoichiometric hydrogen fraction, but the degree of three-dimensional hydrogen cage connectivity and the associated phonon softening/hardening balance that keeps the Eliashberg coupling strong while maintaining structural stability.

**Test:** A comparative synthesis-and-transport program on a set of isostructural or near-isostructural hydrides (for example, La/Y/Ca-based superhydrides and chemically substituted variants) will show that samples with the shortest average H-H distances and highest coordination of hydrogen cages retain a superconducting transition above 100 K at pressures at least 30-50% lower than analogs with weaker cage connectivity. This should correlate with inelastic x-ray/neutron scattering or Raman measurements showing hydrogen-dominated phonon spectra that remain hard and strongly coupled under reduced pressure; if the same compounds either lose superconductivity, fall below 100 K, or undergo a structural transition before that pressure reduction, the hypothesis is refuted.

**Confidence:** 0.66

**Addresses:** Can any hydride family be pushed from megabar pressures toward practical ambient or near-ambient operation without losing the high-Tc phonon-driven state, and what structural motifs best enable that?

### Hypothesis 3

**Claim:** Infinite-layer nickelates and pressure-stabilized Ruddlesden–Popper nickelates are governed by the same pairing glue: antiferromagnetic superexchange in an effectively two-orbital Ni 3d_{x^2-y^2}/3d_{z^2} model, with the observed differences in Tc and phase diagrams arising mainly from the relative strength of self-doping and interlayer hybridization rather than a change in pairing symmetry.

**Mechanism:** In both families, superconductivity should emerge when NiO2-derived bands are close enough to a Hund/superexchange-driven spin-fluctuation instability that singlet pairing is enhanced by short-range antiferromagnetic exchange. Infinite-layer compounds realize a quasi-2D, more weakly hybridized limit with stronger self-doping from rare-earth/axial bands, while Ruddlesden–Popper compounds add interlayer hopping and explicit bilayer splitting that renormalize the same underlying pairing channel. If this is correct, changing the degree of interlayer coupling should move Tc and gap anisotropy continuously but should not change the dominant gap symmetry or the momentum structure of the pairing interaction.

**Test:** After compensating for carrier density, a systematic pressure- or strain-tuning experiment should find the same leading superconducting gap symmetry and the same spin-fluctuation energy scale in both families: (i) phase-sensitive Josephson interferometry and angle-resolved quasiparticle interference should detect the same sign-changing even-parity order parameter in infinite-layer and Ruddlesden–Popper nickelates; (ii) inelastic neutron or resonant inelastic x-ray scattering should show that the superconducting Tc scales with the integrated low-energy antiferromagnetic spectral weight in both systems; and (iii) if the mechanism is common, suppressing interlayer hybridization in a Ruddlesden–Popper nickelate by epitaxial strain should not eliminate superconductivity, only lower Tc smoothly. A decisive refutation would be observing a robust change of pairing symmetry or a Tc that correlates with a nonmagnetic interlayer mode in one family but not the other.

**Confidence:** 0.41

**Addresses:** Are infinite-layer and Ruddlesden–Popper nickelates governed by a common pairing mechanism, or are they distinct platforms with different low-energy theories?

### Hypothesis 4

**Claim:** In magic-angle twisted bilayer/trilayer graphene, the superconducting state is predominantly valley-singlet, spin-singlet chiral d+id, and its pairing glue is mainly electronic via exchange of short-range collective fluctuations of the correlated normal state rather than conventional phonons.

**Mechanism:** Near the magic angle, the flat bands amplify Coulomb interactions and quantum-geometric effects, making intervalley and intravalley scattering by spin/valley/charge fluctuations from proximate correlated insulators the dominant attractive channel after projection into the narrow-band manifold. This favors a d-wave-like basis that can lower interaction energy on the moiré lattice, and a chiral d+id combination is selected because it gaps the Fermi surface more fully and couples naturally to the two-component lattice-irrep structure. Phonons may renormalize Tc but are not the primary source of the pairing kernel, so isotope effects should be weak and the superconducting gap anisotropy should track the symmetry of the correlated electronic background rather than lattice vibrational spectra.

**Test:** Two experiments should discriminate this hypothesis: (1) phase-sensitive Josephson/interferometric measurements on symmetry-engineered junctions should detect a two-component d-wave order parameter with a chiral d+id state below Tc, including a spontaneous time-reversal-symmetry-breaking signal (e.g., Kerr effect or edge currents) that onsets with superconductivity; (2) isotope substitution of carbon with 13C should produce only a small Tc shift and no corresponding change in the pairing symmetry, whereas tuning toward or away from nearby correlated insulators should strongly modulate Tc and gap magnitude if collective electronic fluctuations are the main glue.

**Confidence:** 0.55

**Addresses:** What is the true superconducting order parameter in magic-angle graphene systems, and is the pairing primarily phononic, electronic, or mediated by collective modes of the correlated normal state?

### Hypothesis 5

**Claim:** A single predictive design principle exists across cuprates, nickelates, hydrides, and moiré superconductors: the superconducting transition temperature is maximized when the low-energy electronic states have a large pairing susceptibility generated by a high density of pairable modes, but Tc is suppressed when the same electronic manifold also hosts a nearby symmetry-breaking instability; quantitatively, the best family-specific descriptors are the ratio of pairing-scale enhancement to the leading competing-order susceptibility and the degree of Fermi-surface/quantum-geometry nesting into that instability channel.

**Mechanism:** In all four platforms, superconductivity emerges from a narrow set of low-energy states that are amplified by structure: CuO2 or NiO2 planes, hydrogen clathrate cages, or moiré flat bands. The common mechanism is that lattice geometry and orbital composition set both (i) the pairing kernel—via strong electron-phonon coupling in hydrides, spin/charge fluctuations in cuprates and nickelates, or quantum-geometric/interaction effects in moiré systems—and (ii) a competing order parameter with nearly the same phase space (charge order, magnetism, structural distortion, or flavor polarization). Tc is therefore controlled not by the absolute coupling strength alone, but by how effectively the structural/electronic design enhances pairing while avoiding a near-degeneracy with the dominant competing order. If this framework is correct, then materials with similar values of the dimensionless ratio pairing-susceptibility / competing-order-susceptibility should exhibit comparable optimized Tc even across very different microscopic Hamiltonians.

**Test:** Across a curated set of representative compounds/devices in the four families, independently measured pairing susceptibilities (from isotope shift and phonon softening in hydrides, spin-fluctuation spectral weight and superfluid stiffness in cuprates/nickelates, and tunneling/quantum-oscillation pairing proxies in moiré systems) and leading competing-order susceptibilities (charge density wave, magnetism, or flavor-polarization/structural instability) will collapse onto a single monotonic Tc trend when plotted as Tc versus the ratio χ_pair/χ_compete; specifically, the highest-Tc samples in each family will cluster near the same critical window of this ratio, while samples tuned to increase χ_compete at fixed χ_pair will show a reproducible Tc suppression of order tens of percent. The hypothesis is falsified if, after controlling for carrier density and dimensionality, no cross-family collapse is observed and Tc cannot be predicted better than chance from the proposed ratio in at least one entire family.

**Confidence:** 0.63

**Addresses:** Can one formulate a predictive materials-design framework that spans cuprates, nickelates, hydrides, and moiré systems, or are these families unified only at a phenomenological level?

### Hypothesis 6

**Claim:** In underdoped cuprates, if the pseudogap is primarily a preformed-pair state, then the onset temperature of the pseudogap should coincide with a measurable increase in short-range superconducting phase stiffness and should produce a Bogoliubov-like particle–hole-symmetric gap in momentum-resolved tunneling, whereas competing-order or fractionalized descriptions should not show both signatures simultaneously above Tc.

**Mechanism:** Preformed-pair physics posits that pairing amplitude forms above Tc but global phase coherence is destroyed by fluctuations; this should leave local superconducting spectral fingerprints and enhanced diamagnetic/phase-stiffness response without long-range coherence. By contrast, a competing order gaps the Fermi surface through a reconstruction mechanism tied to a broken symmetry (charge, spin, nematic), which generically yields asymmetric band folding or pocket reconstruction rather than Bogoliubov symmetry. A fractionalized/Fermi-surface-reconstruction scenario can also gap antinodal states, but it need not generate local pairing coherence or a superconducting-like symmetric dispersion above Tc.

**Test:** On the same underdoped sample, using temperature-dependent Josephson STM or phase-sensitive THz conductivity together with momentum-resolved tunneling/ARPES, the pseudogap onset temperature T* should either (i) coincide with a detectable above-Tc enhancement of pair susceptibility/short-range phase stiffness and a particle–hole-symmetric Bogoliubov dispersion, supporting preformed pairs, or (ii) show Fermi-surface reconstruction signatures (backfolded bands, small pockets, broken-symmetry order parameter) without any above-Tc phase-stiffness enhancement, refuting the preformed-pair hypothesis. A decisive falsification is observing a robust pseudogap with no above-Tc pairing signatures but clear symmetry-breaking reconstruction at T* across doping.

**Confidence:** 0.67

**Addresses:** Which experimentally accessible observables can decisively distinguish preformed-pair, competing-order, and fractionalized/Fermi-surface-reconstruction descriptions of the cuprate pseudogap?

There are multiple hypotheses because each open question was processed by one theorist — and this is the basic mechanism of hierarchical multi-agent orchestration. One agent produces structured output, the schema defines the interface, and downstream agents consume it programmatically. No string parsing, no prompt engineering the handoff. The types *are* the protocol.

## Up next

In the next tutorial, we will introduce the concept of events in AG2 and dive deeper into multi-agent orchestration.

- [Day 1: Simple search agent with Exa](01_get_started.ipynb)
- [Day 2: Structured output with response_schema](02_typed_findings.ipynb)
- For more info on AG2 beta, read the [docs](https://docs.ag2.ai/)